In [ ]:
import keras_hub

tokenizer= keras_hub.models.Tokenizer.from_preset("roberta_base_en")
backbone= keras_hub.models.Backbone.from_preset("roberta_base_en")

In [ ]:
backbone.summary()

In [ ]:
import os, pathlib, shutil, random
import keras

zip_path= keras.utils.get_file(
    origin= "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz",
    extract=True,
)
imdb_extract_dir= pathlib.Path(zip_path)/ "aclImdb"
train_dir= pathlib.Path("imdb_train")
test_dir= pathlib.Path("imdb_test")
val_dir= pathlib.Path("imdb_val")

shutil.copytree(imdb_extract_dir/"test",test_dir)

val_percentage= 0.2
for category in ("neg","pos"):
    src_dir= imdb_extract_dir/"train"/category
    src_files= os.listdir(src_dir)
    random.Random(1337).shuffle(src_files)
    num_val_samples= int(len(src_files)*val_percentage)

    os.makedirs(val_dir/category)
    for file in src_files[:num_val_samples]:
        shutil.copy(src_dir/file, val_dir/category/file)
    os.makedirs(train_dir/category)
    for file in src_files[num_val_samples:]:
        shutil.copy(src_dir/file, train_dir/category/file)

from keras.utils import text_dataset_from_directory

batch_size=32
train_ds= text_dataset_from_directory(train_dir, batch_size=batch_size)
val_ds= text_dataset_from_directory(val_dir, batch_size=batch_size)
test_ds= text_dataset_from_directory(test_dir, batch_size=batch_size)

In [ ]:
def preprocess(text,label):
    packer= keras_hub.layers.StartEndPacker(
        sequence_length=512,
        start_value= tokenizer.start_token_id,
        end_value= tokenizer.end_token_id,
        pad_value= tokenizer.pad_token_id,
        return_padding_mask=True,
    )
    token_ids, padding_mask= packer(tokenizer(text))
    return {"token_ids": token_ids, "padding_mask": padding_mask},label

preprocessed_train_ds = train_ds.map(preprocess)
preprocessed_val_ds = val_ds.map(preprocess)
preprocessed_test_ds = test_ds.map(preprocess)

In [ ]:
next(iter(preprocessed_train_ds))

In [ ]:
from keras import layers

In [ ]:
inputs= backbone.input
x= backbone(inputs)
x=x[:,0,:]
x=layers.Dropout(0.1)(x)
x= layers.Dense(768, activation="relu")(x)
x= layers.Dropout(0.1)(x)
outputs= layers.Dense(1,activation="sigmoid")(x)
classifier= keras.Model(inputs,outputs)

In [ ]:
classifier.compile(
    optimizer= keras.optimizers.Adam(5e-5),
    loss= "binary_crossentropy",
    metrics=["accuracy"],
)
classifier.fit(
    preprocessed_train_ds,
    validation_data= preprocessed_val_ds,
)

In [ ]:
classifier.evaluate(preprocessed_test_ds)